# B92 Quantum Key Distribution

B92 is a simplified QKD protocol using only two non-orthogonal states:
- Bit 0 is encoded as $|0\rangle$ (Z basis)
- Bit 1 is encoded as $|+\rangle$ (X basis)

Bob keeps only conclusive results (where he measured $|1\rangle$), discarding the rest.

In [ ]:
import qsharp
import matplotlib.pyplot as plt
import random

In [ ]:
qsharp.init()
with open("Program.qs") as f:
    qsharp.eval(f.read())

from qsharp.code import B92PrepareAndMeasure, B92WithEavesdropper

In [ ]:
def run_b92(n_bits, eavesdropper_prob=0.0):
    alice_values = [random.choice([True, False]) for _ in range(n_bits)]
    bob_bases = [random.choice([True, False]) for _ in range(n_bits)]

    bob_results = []
    for i in range(n_bits):
        if eavesdropper_prob > 0.0 and random.random() < eavesdropper_prob:
            eve_basis = random.choice([True, False])
            r = B92WithEavesdropper(alice_values[i], eve_basis, bob_bases[i])
        else:
            r = B92PrepareAndMeasure(alice_values[i], bob_bases[i])
        bob_results.append(str(r) == "One")

    # B92 reconciliation: Bob keeps only conclusive results (measured One)
    alice_key = []
    bob_key = []
    for i in range(n_bits):
        if bob_results[i]:  # Bob got a conclusive result
            alice_key.append(alice_values[i])
            bob_key.append(not bob_bases[i])

    # Eavesdropper check
    check_bits = []
    final_alice = []
    final_bob = []
    for i in range(len(alice_key)):
        if random.random() < 0.5:
            check_bits.append(alice_key[i] != bob_key[i])
        else:
            final_alice.append(alice_key[i])
            final_bob.append(bob_key[i])

    error_rate = sum(check_bits) / len(check_bits) if check_bits else 0.0
    keys_match = final_alice == final_bob

    return {
        "key_length": len(final_alice),
        "error_rate": error_rate,
        "keys_match": keys_match,
    }

In [ ]:
scenarios = [
    ("No eavesdropper", 0.0),
    ("50% eavesdropping", 0.5),
    ("100% eavesdropping", 1.0),
]

error_rates = []
for name, prob in scenarios:
    result = run_b92(1024, eavesdropper_prob=prob)
    error_rates.append(result["error_rate"])
    status = "MATCH" if result["keys_match"] else "MISMATCH"
    detected = "Eavesdropper detected!" if result["error_rate"] > 0 else "No eavesdropper"
    print(f"{name}: error rate = {result['error_rate']*100:.1f}%, key length = {result['key_length']}, keys {status}. {detected}")

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar([s[0] for s in scenarios], [r * 100 for r in error_rates])
ax.set_ylabel("Error rate (%)")
ax.set_ylim(0, 30)
plt.title("B92 - Error Rate vs Eavesdropping")
plt.show()